# core

> What every tool result looks like, and the limits it is held to.

In [ ]:
#| default_exp core

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import tempfile
from tempfile import TemporaryDirectory
from fastcore.test import test_eq, test_fail


In [ ]:
#| export
import json, os
from fastcore.basics import AttrDict, first, listify
from fastcore.foundation import L
from fastcore.xtras import Path, str_diff

## Limits

Tool results consume model context. `MAX_TOOL_CHARS` limits one result. Factories accept the limit
as `mx` because the model sets the budget.

In [ ]:
#| export
MAX_TOOL_CHARS = 6000   # chars per tool result, budgeted for the smallest model
MAX_HITS = 20           # search hits one result carries
MAX_GREP_HITS = 60      # exact matches one `grep` returns. Part of the host contract
MAX_API = 200           # public names one `public_api` listing returns. Part of the same contract
MAX_FILE = 2_000_000    # bytes. A file larger than this is data, not source

In [ ]:
# a grep answers with more rows than a search, because an exact match is cheaper to read
assert MAX_GREP_HITS > MAX_HITS
assert MAX_TOOL_CHARS < 8_000 and MAX_FILE > MAX_TOOL_CHARS
test_eq((MAX_TOOL_CHARS, MAX_HITS, MAX_GREP_HITS, MAX_API, MAX_FILE), (6000, 20, 60, 200, 2_000_000))

Every search backend returns a `Hit` with the path, line, symbol and matching text.

In [ ]:
#| export
class Hit(AttrDict):
    "One search hit: path, line, symbol, text."
    def __init__(self, path, line=1, symbol='', text=''): super().__init__(path=path, line=line, symbol=symbol, text=text)
    def __repr__(self): return f'{self.path}:{self.line}  {self.symbol}  {self.text}'

In [ ]:
h = Hit('shalya/core.py', 12, 'clip', 'def clip(s, n=MAX_TOOL_CHARS, more=\'\'):')
h.path, h.line, h.symbol

('shalya/core.py', 12, 'clip')

In [ ]:
test_eq(repr(h), "shalya/core.py:12  clip  def clip(s, n=MAX_TOOL_CHARS, more=''):")

## Failure

Tool failures start with `ERROR: `. `failed` checks this prefix. Returning `err(...)` lets the model
inspect a failure without ending the turn.

In [ ]:
#| export
ERR = 'ERROR: '

class HostError(Exception): "A host refusal."

def host_err(e):
    "A caught exception, for a user-facing surface."
    return f'{type(e).__name__}: {e}'

def err(what, e=None):
    "One tool failure, spelled the way every other tool spells it."
    return f'{ERR}{what}' + (f': {host_err(e)}' if e is not None else '')

def failed(result):
    "Whether a tool result starts with `ERROR: `."
    return str(result or '').startswith(ERR)

In [ ]:
try: raise HostError('path is outside the open folders: /etc/passwd')
except HostError as e: shown = host_err(e)
shown

'HostError: path is outside the open folders: /etc/passwd'

In [ ]:
test_eq(shown, 'HostError: path is outside the open folders: /etc/passwd')
test_eq(host_err(KeyError('nope')), "KeyError: 'nope'")
test_eq(err('could not read', HostError('no such file')), 'ERROR: could not read: HostError: no such file')
test_eq(ERR, 'ERROR: ')

In [ ]:
try: json.loads('{not json}')
except Exception as e: msg = err('cannot read the edits', e)
msg

'ERROR: cannot read the edits: JSONDecodeError: Expecting property name enclosed in double quotes: line 1 column 2 (char 1)'

In [ ]:
test_eq(failed(msg), True)
test_eq(failed('ERROR was not the problem'), False)
test_eq(err('no such file'), 'ERROR: no such file')

## The sandbox

Every file tool resolves its path first and asks about it afterwards, so a `..`, a `~` or a symlink
is judged by where it lands rather than by how it was spelled. `Sandbox` is the one place that
judges, and `Unsafe` is what it raises. A host holds one instead of writing the rule again.

In [ ]:
#| export
SANDBOX = 'path is outside the open folders'
SECRET = 'path holds credentials and is never read'
NO_ROOTS = 'no folders are open, so no path is inside them'

#: Credential-shaped paths refused even with `read_outside`. `fnmatch` on the resolved path.
DENY = ('*/.ssh/*', '*/.aws/*', '*/.gnupg/*', '*/.config/gcloud/*', '*/.netrc',
        '*/.git-credentials', '*/.codex/auth.json', '*/.claude/.credentials.json',
        '*/.env', '*/.env.*', '*/id_rsa*', '*/id_ed25519*', '*.pem', '*.key', '*.p12')

class Unsafe(HostError): "A path that resolves outside every open root, or one refused for safety."

def denied(path, patterns=DENY):
    "Whether `path` matches a refused credential path."
    from fnmatch import fnmatch
    s = Path(path).as_posix()
    return any(fnmatch(s, pat) for pat in patterns)

`denied` is what `read_outside` still refuses: credential-shaped paths, matched against the resolved
path. A read tool is not a way to exfiltrate a key.

In [ ]:
[p for p in ('/home/k/.ssh/id_rsa', '/proj/.env', '/proj/key.pem', '/proj/app.py') if denied(p)]

In [ ]:
#| hide
for p in ('/home/k/.ssh/id_rsa', '/home/k/.aws/credentials', '/proj/.env', '/proj/.env.local',
          '/proj/id_ed25519', '/proj/key.pem', '/proj/cert.p12', '/home/k/.netrc'):
    assert denied(p), p
for p in ('/proj/app.py', '/proj/README.md', '/proj/environment.yml'):
    assert not denied(p), p
test_eq(denied('/proj/.env', patterns=()), False)         # an empty deny list denies nothing
assert issubclass(Unsafe, HostError)                      # `except HostError` still catches a refusal

In [ ]:
#| export
class Sandbox:
    "The open folders, and the one place that decides whether a path is inside them."
    def __init__(self,
                 roots=(),             # folders to open; each must already exist
                 read_outside=False,   # let reads name any path on this machine
                 deny=DENY):           # what `read_outside` still refuses to open
        self.roots, self.read_outside, self.deny = L(), bool(read_outside), tuple(deny or ())
        for r in roots: self.add_root(r)
    def __repr__(self): return f'Sandbox({[str(r) for r in self.roots]})'
    def covers(self, p):
        "Whether `p` is inside an open root, or is one of them."
        return any(Path(p).is_relative_to(r) for r in self.roots)
    def add_root(self, p):
        "Open a folder and close any open root inside it. Returns the root that now covers it."
        p = Path(p).expanduser().resolve()
        if not p.exists(): raise Unsafe(f'no such folder: {p}')
        if not p.is_dir(): raise Unsafe(f'not a folder, so it cannot be a root: {p}')
        if self.covers(p): return self.root_of(p)
        self.roots = L([r for r in self.roots if p not in r.parents]) + [p]
        return p
    def drop_root(self, p):
        "Close a folder. Nothing under it resolves any more."
        p = Path(p).expanduser().resolve()
        self.roots = self.roots.filter(lambda r: r != p)
        return p
    def check(self, path, must_exist=False, reading=False):
        "Resolve `path`, and refuse it unless it lands inside an open root."
        if not self.roots: raise Unsafe(f'{NO_ROOTS}: {path}')
        p = Path(path).expanduser()
        if not p.is_absolute(): p = self.roots[0]/p
        p = p.resolve()
        if not self.covers(p):
            if not (reading and self.read_outside): raise Unsafe(f'{SANDBOX}: {p}')
            if denied(p, self.deny): raise Unsafe(f'{SECRET}: {p}')
        if must_exist and not p.exists(): raise Unsafe(f'no such file: {p}')
        return p
    def root_of(self, path):
        "The open root holding `path`, or None when `read_outside` let it through from elsewhere."
        p = self.check(path, reading=True)
        return first(r for r in self.roots if p.is_relative_to(r))
    def bare(self, name):
        "A name typed by a person or a model, refused unless it is one path component."
        if '/' in name or os.sep in name or name in ('', '.', '..'): raise Unsafe(f'{name!r} is not a bare filename')
        return name
    def under(self, path, name):
        "Where an entry called `name` goes inside `path`. `name` is one component, never a path."
        return self.check(self.check(path)/self.bare(name))

A path that does not exist yet still resolves, because that is what writing a new file asks for.
`must_exist=True` is how a reader says otherwise.

In [ ]:
tmp = TemporaryDirectory()
root = Path(tmp.name).resolve()/'proj'
(root/'pkg').mkdir(parents=True)
(root/'pkg'/'a.py').write_text('x = 1\n')

sb = Sandbox([root])
sb.check('pkg/a.py').name, sb.check('pkg/new.py').name, sb.covers(root/'pkg'), sb.root_of('pkg/a.py') == root

The ways out of a sandbox, and the answer to each. A symlink escapes only if `covers` is asked
before resolution, which is why `check` resolves first. With no folders open nothing resolves at
all, so a host that has not opened anything cannot be talked into reading `/etc/passwd`.

In [ ]:
outside = Path(tmp.name).resolve()/'outside'
outside.mkdir()
(outside/'secret.txt').write_text('token\n')
(root/'link').symlink_to(outside, target_is_directory=True)

test_fail(lambda: sb.check('link/secret.txt'), contains=SANDBOX)        # a symlink out
test_fail(lambda: sb.check('../outside/secret.txt'), contains=SANDBOX)  # .. traversal
test_fail(lambda: sb.check(outside/'secret.txt'), contains=SANDBOX)     # an absolute path elsewhere
test_fail(lambda: sb.check('pkg/new.py', must_exist=True), contains='no such file')
test_fail(lambda: Sandbox().check('/etc/passwd'), contains=NO_ROOTS)    # nothing open, nothing readable

`read_outside` widens reads only. A write still has to land inside an open folder, and a credential
path is refused either way.

In [ ]:
lax = Sandbox([root], read_outside=True)
lax.check(outside/'secret.txt', reading=True).name

In [ ]:
#| hide
test_fail(lambda: lax.check(outside/'secret.txt'), contains=SANDBOX)          # a write never leaves
test_fail(lambda: lax.check(outside/'.env', reading=True), contains=SECRET)
test_fail(lambda: lax.check('/home/k/.ssh/id_rsa', reading=True), contains=SECRET)
test_eq(Sandbox([root], read_outside=True, deny=()).check(outside/'.env', reading=True).name, '.env')

Open folders never nest: opening a parent closes the roots inside it, and opening something already
covered adds nothing. Without that, one file would be walked and indexed once per root that holds it.

In [ ]:
sb2 = Sandbox([root/'pkg', root])                         # a folder and its parent, opened together
[r.name for r in sb2.roots]

`bare` is for the other half of the problem: a model asked for a *name* and handed over a path.
`under` puts one inside a folder without ever letting it climb out of it.

In [ ]:
sb.under('pkg', 'b.py').name, sb.bare('b.py')

In [ ]:
#| hide
test_eq(sb.under('pkg', 'b.py'), root/'pkg'/'b.py')
test_fail(lambda: sb.under('pkg', '../../etc'), contains='not a bare filename')
test_fail(lambda: sb.under('pkg', '..'), contains='not a bare filename')
test_fail(lambda: sb.bare(''), contains='not a bare filename')

In [ ]:
#| hide
test_eq(sb2.add_root(root/'pkg'), root)                   # already covered: the root that holds it
test_eq(len(sb2.roots), 1)
sb2.drop_root(root)
test_eq(list(sb2.roots), [])
test_fail(lambda: Sandbox([root/'pkg'/'a.py']), contains='not a folder')
test_fail(lambda: Sandbox([root/'nope']), contains='no such folder')
test_eq(sb.root_of(root), root)
test_eq(lax.root_of(outside/'secret.txt'), None)          # read from outside, so no root holds it
tmp.cleanup()

## Clipping

`clip` truncates at a line boundary. `clip_lines` also reports the next line, which supports paging.

In [ ]:
#| export
def clip(s, n=MAX_TOOL_CHARS, more=''):
    "Truncate a tool result to `n` chars. A caller with a way to resume passes it as `more`."
    s = str(s)
    if len(s) <= n: return s
    cut = s[:n]
    nl = cut.rfind('\n')
    if nl > n * 0.6: cut = cut[:nl]
    note = f'[truncated: {len(cut)} of {len(s)} chars shown'
    return cut + f'…{note}. {more}]' if more else cut + f'…{note}]'

In [ ]:
print(clip('one\ntwo\nthree\nfour\nfive\n', 12))

one
two
thre…[truncated: 12 of 24 chars shown]


In [ ]:
test_eq(clip('short', 12), 'short')
test_eq(clip('one\ntwo\nthree\nfour', 12).splitlines()[0], 'one')

A line longer than the budget is truncated by character count. The notice reports the original and
shown lengths without offering the same line as a resume point.

In [ ]:
#| export
def clip_lines(lines, start=1, n=MAX_TOOL_CHARS, more='', empty='(nothing)'):
    "Render `lines` within the budget, and say which line to resume from."
    lines = listify(lines)
    if not lines: return empty
    out, used = [], 0
    for i, line in enumerate(lines):
        line = str(line)
        if used + len(line) + 1 > n:
            if out:
                rest = len(lines) - i
                tail = f'\n…[{rest} more line(s) not shown'
                hint = more.format(next=start + i) if '{next}' in more else more
                return '\n'.join(out) + (f'{tail}. {hint}]' if hint else f'{tail}]')
            keep, rest = max(1, n - 1), len(lines) - 1
            more_lines = f', and {rest} more line(s) not shown' if rest else ''
            return line[:keep] + f'\n…[line {start} is {len(line)} chars; {keep} shown{more_lines}]'
        out.append(line); used += len(line) + 1
    return '\n'.join(out)

In [ ]:
print(clip_lines(['def a(): pass', 'def b(): pass', 'def c(): pass'], n=30,
                more='read from line {next}'))

def a(): pass
def b(): pass
…[1 more line(s) not shown. read from line 3]


In [ ]:
test_eq(clip_lines([], empty='(no matches)'), '(no matches)')
test_eq(clip_lines(['a', 'b'], n=100), 'a\nb')
long = clip_lines(['x' * 50], n=20)
assert long.startswith('x' * 19) and '50 chars' in long, long

## Edits

Tool calls carry edits as JSON. `cmds` parses exhash commands. `edits` parses exact-text
replacements. Both reject ambiguous input with an actionable error.

In [ ]:
#| export
def cmds(commands):
    "Models emit JSON and exhash wants tuples, nested ones included: `[[...]]` becomes `[(...)]`."
    if isinstance(commands, str): commands = json.loads(commands)
    if not isinstance(commands, list): raise ValueError('commands must be a JSON list of command arrays')
    def _t(c):
        if not isinstance(c, (list, tuple)): raise ValueError(f'each command must be an array, got {type(c).__name__}')
        return tuple(_t(x) if isinstance(x, (list, tuple)) else x for x in c)
    return [_t(c) for c in commands]

In [ ]:
cmds('[["12|a1b2|", "s", "old", "new"], ["30|9f3c|", "a", "appended"]]')

[('12|a1b2|', 's', 'old', 'new'), ('30|9f3c|', 'a', 'appended')]

In [ ]:
test_eq(cmds([['1|ab|', 'd']]), [('1|ab|', 'd')])
test_fail(lambda: cmds('{"not": "a list"}'), contains='must be a JSON list')
test_fail(lambda: cmds('["bare string"]'), contains='must be an array')

`edits` takes the exact-text form in any of its three spellings. `apply_edits` holds the refusals.
Each one names the edit that is wrong and what to do about it. A match that is not unique needs
more context. Two edits over the same span need merging.

In [ ]:
#| export
def edits(es):
    "A JSON string, `{'oldText','newText'}` dicts, or `[old, new]` pairs: all three are unambiguous."
    if isinstance(es, str): es = json.loads(es)
    if isinstance(es, dict): es = [es]
    if not isinstance(es, (list, tuple)): raise ValueError('edits must be a JSON array')
    out = []
    for e in es:
        if isinstance(e, dict):
            if 'oldText' not in e or 'newText' not in e: raise ValueError("each edit needs 'oldText' and 'newText'")
            out.append((str(e['oldText']), str(e['newText'])))
        elif isinstance(e, (list, tuple)) and len(e) == 2: out.append((str(e[0]), str(e[1])))
        else: raise ValueError('each edit must be {"oldText":…,"newText":…} or [old, new]')
    return out

def apply_edits(text, es):
    "Apply exact-text edits to `text`, or raise saying which one is wrong and why."
    spans = []
    for i, (old, new) in enumerate(es, 1):
        if not old: raise ValueError(f'edit {i}: oldText is empty; use create_file to write a whole file')
        n = text.count(old)
        if n == 0: raise ValueError(f'edit {i}: oldText not found. It must match the file exactly, including indentation. Re-read the file and try again')
        if n > 1: raise ValueError(f'edit {i}: oldText matches {n} places. Include more surrounding lines so it matches exactly one')
        at = text.index(old)
        spans.append((at, at + len(old), new, i))
    spans.sort()
    for (s1, e1, _, i1), (s2, _, _, i2) in zip(spans, spans[1:]):
        if s2 < e1: raise ValueError(f'edits {i1} and {i2} overlap; merge them into one edit')
    out, at = [], 0
    for s, e, new, _ in spans:
        out.append(text[at:s])
        out.append(new)
        at = e
    out.append(text[at:])
    return ''.join(out)

In [ ]:
before = 'def greet(name):\n    return "hello " + name\n'
after = apply_edits(before, edits('[{"oldText": "hello", "newText": "howdy"}]'))
after

'def greet(name):\n    return "howdy " + name\n'

In [ ]:
test_eq(edits([['a', 'b']]), [('a', 'b')])
test_eq(edits({'oldText': 'a', 'newText': 'b'}), [('a', 'b')])
test_eq(after, 'def greet(name):\n    return "howdy " + name\n')

The three refusals, on real text. Each message tells the model what to change rather than that it
failed.

In [ ]:
test_fail(lambda: apply_edits(before, [('name', 'who')]), contains='matches 2 places')
test_fail(lambda: apply_edits(before, [('nowhere', 'x')]), contains='not found')
test_fail(lambda: apply_edits(before, [('def greet', 'def hi'), ('greet(name)', 'hi(n)')]),
          contains='overlap')

Edit approvals carry a unified diff.

In [ ]:
#| export
def diff_text(before, after, path='file'):
    "Return a unified diff."
    return str_diff(before, after, n=2, names=(f'a/{path}', f'b/{path}'))

In [ ]:
print(diff_text(before, after, 'greet.py'))

--- a/greet.py
+++ b/greet.py
@@ -1,2 +1,2 @@
 def greet(name):
-    return "hello " + name
+    return "howdy " + name


In [ ]:
assert '-    return "hello " + name' in diff_text(before, after)
assert diff_text(before, before) == ''

## Which tools write

Writes require approval. `@writes` marks a callable; `WRITE_TOOLS` provides the same fact when a
caller has only its name. The tests require both representations to agree.

In [ ]:
#| export
def writes(f):
    "Mark a tool as one that changes something. `Approvals` puts these in front of a person."
    f.writes = True
    return f

def is_write(t):
    "Whether `t` is a tool that changes something."
    return bool(getattr(t, 'writes', False))

def acts(f):
    "Mark a tool that acts without writing a file the user owns."
    f.acts = True
    return f

def has_effect(t):
    "Whether `t` acts. Orthogonal to `is_write`: these are the effects approval does not gate."
    return bool(getattr(t, 'acts', False))

def one_line(v, n=90):
    "One line of a value, short enough to sit in a list."
    t = ' '.join(str(v or '').split())
    return t if len(t) <= n else t[:n - 1] + '…'

SUMMARIES = {}
def summary(fn):
    "Mark the one line a person reads after this tool runs. `fn` is given the call's arguments."
    def _mark(t):
        t.summary = fn
        SUMMARIES[t.__name__] = fn
        return t
    return _mark

def summarise(tool, args=None):
    "The imperative one-liner for a call: what a person would say they just did."
    a = args if isinstance(args, dict) else {}
    nm = tool if isinstance(tool, str) else getattr(tool, '__name__', '')
    fn = getattr(tool, 'summary', None) or SUMMARIES.get(nm)
    if fn is not None:
        try: return fn(a)
        except Exception: pass
    return f'{nm}({", ".join(f"{k}={one_line(v, 30)!r}" for k, v in a.items())})'

GIT_READ_TOOLS = ('git_status', 'git_divergence', 'git_rebase_preview')
GIT_WRITE_TOOLS = frozenset({'git_remote', 'git_checkout'})
GIT_TOOLS = (*GIT_READ_TOOLS, *sorted(GIT_WRITE_TOOLS))

WRITE_TOOLS = frozenset({'edit_file', 'replace_text', 'ast_edit', 'create_file', 'edit_cell', 'add_cell', 'run_python', 'run_shell', 'memory_forget',
                         'create_skill', 'cancel_watch', 'add_root'}) | GIT_WRITE_TOOLS

ACTING_TOOLS = frozenset({'inspect_python', 'api_call', 'generate_image', 'research', 'watch_url', 'set_reminder'})

In [ ]:
@writes
def create_file(path, text): return f'wrote {path}'
def view_file(path): return 'contents'
is_write(create_file), is_write(view_file)

(True, False)

In [ ]:
test_eq(is_write(create_file), True)
test_eq(is_write(view_file), False)
test_eq(create_file.__name__ in WRITE_TOOLS, True)
test_eq(sorted(GIT_TOOLS), sorted(set(GIT_READ_TOOLS) | GIT_WRITE_TOOLS))

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()